# Podcast episode → audio (VibeVoice, free on Colab)

**Before running:** Runtime → Change runtime type → **GPU (T4 is fine)**.

Then Runtime → **Run all**. Fill in the form in the next cell first. The finished MP3 lands in your Google Drive under `Podcasts/`.

Rough timing: setup ~5 min, then generation is around real-time on a T4 (a 45-minute episode ≈ 45–90 min). Keep the tab open on your phone or it may disconnect.

In [ ]:
#@title 1. Settings
GITHUB_REPO   = "Luke11038/Podcasts" #@param {type:"string"}
GITHUB_BRANCH = "main"                       #@param {type:"string"}
EPISODE_FILE  = "episodes/episode-02-the-late-twenties-squeeze.md" #@param {type:"string"}

# Voice preset for each speaker, in order of first appearance in the script.
# Built-in presets: Alice (f), Maya (f), Carter (m), Frank (m), Mary (f, has bgm - avoid).
# To use your own Aussie voices, upload 10-30s WAV samples to demo/voices/ named e.g. en-Nick_man.wav
# and use "Nick" here.
VOICE_SPEAKER_1 = "Carter" #@param {type:"string"}
VOICE_SPEAKER_2 = "Alice"  #@param {type:"string"}
VOICE_SPEAKER_3 = "Frank"  #@param {type:"string"}
VOICE_SPEAKER_4 = "Maya"   #@param {type:"string"}

MODEL = "microsoft/VibeVoice-1.5B" #@param ["microsoft/VibeVoice-1.5B", "aoi-ot/VibeVoice-Large"]
CFG_SCALE = 1.3 #@param {type:"number"}


In [ ]:
#@title 2. Install VibeVoice (community fork) + ffmpeg
%cd /content
!git clone -q https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!pip install -q -e . 2>&1 | tail -1
!apt-get -qq install -y ffmpeg > /dev/null
print("Installed.")


In [ ]:
#@title 3. Pull the script from GitHub and convert it
import os, re, subprocess, urllib.request
# Works for public repos as-is. For a PRIVATE repo, add a Colab secret named GITHUB_TOKEN
# (key icon in the left sidebar) containing a fine-grained PAT with Contents: read on this repo.
token = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass
url = f"https://api.github.com/repos/{GITHUB_REPO}/contents/{EPISODE_FILE}?ref={GITHUB_BRANCH}"
hdr = {"Accept": "application/vnd.github.raw+json"}
if token: hdr["Authorization"] = f"Bearer {token}"
os.makedirs("/content/episode", exist_ok=True)
with urllib.request.urlopen(urllib.request.Request(url, headers=hdr)) as r:
    open("/content/episode/script.md", "wb").write(r.read())

STOP_AT = "## RESEARCH NOTES"
LABEL = re.compile(r"^([A-Z][A-Z]+):\s*(.+)$")
CUE = re.compile(r"\s*\[[^\]]*\]\s*")
speakers, lines = [], []
for line in open("/content/episode/script.md", encoding="utf-8"):
    line = line.strip()
    if line.startswith(STOP_AT): break
    m = LABEL.match(line)
    if not m: continue
    name, text = m.group(1), re.sub(r"\s{2,}", " ", CUE.sub(" ", m.group(2))).strip()
    if not text: continue
    if name not in speakers: speakers.append(name)
    lines.append(f"Speaker {speakers.index(name)+1}: {text}")
open("/content/episode/script.txt", "w", encoding="utf-8").write("\n".join(lines) + "\n")
voices = [VOICE_SPEAKER_1, VOICE_SPEAKER_2, VOICE_SPEAKER_3, VOICE_SPEAKER_4][:len(speakers)]
print(f"{len(lines)} turns, {len(speakers)} speakers:")
for s, v in zip(speakers, voices): print(f"  {s} -> {v}")


In [ ]:
#@title 4. Generate audio (this is the slow bit)
%cd /content/VibeVoice
cmd = ["python", "demo/inference_from_file.py",
       "--model_path", MODEL,
       "--txt_path", "/content/episode/script.txt",
       "--speaker_names", *voices,
       "--cfg_scale", str(CFG_SCALE),
       "--output_dir", "/content/episode/out"]
print(" ".join(cmd))
subprocess.run(cmd, check=True)
!ls -la /content/episode/out


In [ ]:
#@title 5. Convert to MP3 and save to Google Drive
import glob, os
from google.colab import drive
drive.mount("/content/drive")
wav = sorted(glob.glob("/content/episode/out/*.wav"))[-1]
name = os.path.splitext(os.path.basename(EPISODE_FILE))[0] + ".mp3"
os.makedirs("/content/drive/MyDrive/Podcasts", exist_ok=True)
out = f"/content/drive/MyDrive/Podcasts/{name}"
!ffmpeg -y -loglevel error -i "{wav}" -codec:a libmp3lame -b:a 96k "{out}"
print("Saved to Google Drive:", out)


## Notes
- **Three male voices?** VibeVoice only ships two English male presets (Carter, Frank). For Nick + Dan + Sam episodes, record or grab a 10–30 s clean WAV for the third voice, upload it to `demo/voices/` in the runtime (or commit it to your repo and copy it across in cell 2), and put its name in Settings.
- **Aussie accents:** presets are American. The fix is the same — your own reference WAVs. A mate reading a paragraph into Voice Memos is enough.
- **Emotion:** VibeVoice takes it from the words and punctuation, so scripts are written without `[laughing]` tags. Any that sneak in are stripped automatically.
- **Talking too fast?** Lower `CFG_SCALE` slightly, or split a long turn into two consecutive turns for the same speaker.
- **Disconnected mid-run?** Re-run; the model is cached so setup is quick the second time.
- **Want it faster / no Colab babysitting?** Replicate hosts `microsoft/vibevoice` as a paid API (cents per run). Same text format.